Data Splitting: 60/20/20 Experiment

Overview

This notebook prepares a second train-validation-test split for the Egocentric Activity Recommendation project.

The purpose is to compare the effects of two data-splitting configurations on recommendation performance:

Experiment 1: 80% training, 10% validation, 10% testing

Experiment 2: 60% training, 20% validation, 20% testing

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project directory
project = Path(
    "/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System"
)

# Display available files in the data folders
for folder_name in ["raw", "processed", "splits"]:
    folder = project / "data" / folder_name

    print(f"\n{folder_name.upper()} FILES:")

    if folder.exists():
        for file in sorted(folder.iterdir()):
            if file.is_file():
                print(" ", file.name)
    else:
        print("  Folder not found")


RAW FILES:
  meta-New_York.json.gz
  review-New_York_10 2.json

PROCESSED FILES:
  activity_dataset.csv

SPLITS FILES:
  test.csv
  train.csv
  validation.csv


In [2]:
import json

review_path = (
    project / "data" / "raw" / "review-New_York_10 2.json"
)

with open(review_path, "r", encoding="utf-8") as file:
    for i in range(3):
        review = json.loads(file.readline())
        print(f"\nReview {i + 1}:")
        print(review)


Review 1:
{'user_id': '113722104692308235141', 'name': 'Alvin Martinez', 'time': 1603494795361, 'rating': 5, 'text': "I'm late to posting this but this store especially this amazing representative made my entire month. Helpful kind attentive everything was handled and she did it with a smile. I need 12 more stores like this one instead of the ones on junction", 'pics': [{'url': ['https://lh5.googleusercontent.com/p/AF1QipMOL-t5pYhZ1mX9DKux9oKTJDFy_6R-3jLfwuEA=w150-h150-k-no-p']}], 'resp': None, 'gmap_id': '0x89c25fc9494dce47:0x6d63c807b59a55'}

Review 2:
{'user_id': '107293441492109320298', 'name': 'Johnnie Jackson', 'time': 1620157037403, 'rating': 1, 'text': 'Very dissatisfied I did not get my phone the phone they had there was damage', 'pics': None, 'resp': {'time': 1620268360920, 'text': 'We pride ourselves on providing an awesome experience and I would love to hear more.'}, 'gmap_id': '0x89c25fc9494dce47:0x6d63c807b59a55'}

Review 3:
{'user_id': '100378585801819400296', 'name': '

In [3]:
# Load the processed activity metadata
activity_path = (
    project / "data" / "processed" / "activity_dataset.csv"
)

activity_df = pd.read_csv(
    activity_path,
    dtype={"gmap_id": "string"}
)

print("Activity dataset shape:", activity_df.shape)

print("\nAvailable columns:")
print(activity_df.columns.tolist())

print("\nUnique activity IDs:", activity_df["gmap_id"].nunique())

print("\nFirst five activities:")
display(activity_df.head())

Activity dataset shape: (86577, 17)

Available columns:
['name', 'address', 'gmap_id', 'description', 'latitude', 'longitude', 'category', 'avg_rating', 'num_of_reviews', 'price', 'hours', 'MISC', 'state', 'relative_results', 'url', 'is_activity', 'activity_groups']

Unique activity IDs: 86354

First five activities:


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,is_activity,activity_groups
0,Chef Chucks Table,"Chef Chucks Table, 6128 NY-89, Ovid, NY 14521",0x89d0be3bcc338707:0x3e3d9bf1247ba191,NaN,42.727924,-76.765099,['Diner'],4.7,3,NaN,"[['Thursday', 'Closed'], ['Friday', 'Closed'],...","{'Service options': ['Delivery'], 'Amenities':...",Permanently closed,"['0x89d0eba9d9c0f817:0x1f127a3049db854d', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,True,['Food & Drink']
1,Arcana Juventa Anti-Aging Spa,"Arcana Juventa Anti-Aging Spa, 14 Brighton 11t...",0x89c244688b7609cd:0x205e2d8d173dd35e,NaN,40.582130,-73.956333,"['Spa', 'Waterproofing company']",4.1,15,NaN,"[['Thursday', '10AM–8PM'], ['Friday', '10AM–8P...",NaN,Permanently closed,"['0x89c2446bdf00743d:0x1a016c5e3c603c88', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,True,['Sports & Recreation']
2,Honeoye Creek Wildlife Management Area,"Honeoye Creek Wildlife Management Area, 9033 A...",0x89d13fd0bc4468e1:0x1a81c620f5086b11,NaN,42.806571,-77.525743,"['Nature preserve', 'Park']",4.7,7,NaN,NaN,NaN,NaN,"['0x89d113b7ef15c121:0xe71429a0a567a625', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,True,['Outdoor & Nature']
3,VeganPro Sweets NYC,NaN,0x89c25923cbb1c20d:0xd9edcb0a146ba104,NaN,40.705649,-73.977940,"['Dessert shop', 'Service establishment']",5.0,13,NaN,"[['Thursday', 'Open 24 hours'], ['Friday', 'Op...",{'From the business': ['Identifies as women-le...,Open 24 hours,NaN,https://www.google.com/maps/place//data=!4m2!3...,True,['Food & Drink']
4,Dunkin',"Dunkin', 4008 Bell Blvd, Queens, NY 11361",0x89c261f60bdf13db:0x38da730e4687a97b,Long-running chain serving signature breakfast...,40.763985,-73.771430,"['Coffee shop', 'Bagel shop', 'Bakery', 'Break...",3.5,8,$,"[['Thursday', '6AM–7PM'], ['Friday', '6AM–7PM'...","{'Service options': ['Delivery', 'Takeout', 'D...",Open ⋅ Closes 7PM,"['0x89c3ab9229879ec3:0x3f4b2b46d7d2c503', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,True,['Food & Drink']


In [4]:
# Inspect the existing 80/10/10 split files
for split_name in ["train", "validation", "test"]:
    split_path = project / "data" / "splits" / f"{split_name}.csv"

    sample = pd.read_csv(
        split_path,
        nrows=5,
        dtype={"user_id": "string", "gmap_id": "string"}
    )

    print(f"\n{split_name.upper()} SPLIT")
    print("Columns:", sample.columns.tolist())
    display(sample.head(3))


TRAIN SPLIT
Columns: ['user_id', 'gmap_id', 'rating', 'time']


,user_id,gmap_id,rating,time
0,100000012795753291799,0x89c25d977025692d:0x378d24e13eb8f34d,4,1512264700675
1,100000012795753291799,0x89c258e28c304997:0xfcafe4e7ce35ee8c,4,1518768933898
2,100000012795753291799,0x89c2590f34e21a53:0x749079380908a830,4,1519751339673



VALIDATION SPLIT
Columns: ['user_id', 'gmap_id', 'rating', 'time']


,user_id,gmap_id,rating,time
0,100000012795753291799,0x89c245caca6f414d:0x9068bde30b6b8efe,4,1581218828680
1,100000036464530353686,0x89c258ff48e75613:0x1de767bc6d9ae0c,5,1557705390025
2,100000036464530353686,0x89c2f662a75ebad5:0x54bfc08ac601aa8d,5,1561338245436



TEST SPLIT
Columns: ['user_id', 'gmap_id', 'rating', 'time']


,user_id,gmap_id,rating,time
0,100000012795753291799,0x89c25a4c43c5a19f:0x3f6401582b49c95c,4,1617057156812
1,100000036464530353686,0x89c26003eb55318d:0x2194c3be6f725bdd,5,1565556386612
2,100000036464530353686,0x89c2601d98a56ca7:0x15cb45ed88ec1a6d,5,1571141702203


In [5]:
# Load all three existing split files
train_80 = pd.read_csv(
    project / "data" / "splits" / "train.csv",
    dtype={"user_id": "string", "gmap_id": "string"}
)

validation_80 = pd.read_csv(
    project / "data" / "splits" / "validation.csv",
    dtype={"user_id": "string", "gmap_id": "string"}
)

test_80 = pd.read_csv(
    project / "data" / "splits" / "test.csv",
    dtype={"user_id": "string", "gmap_id": "string"}
)

# Recombine retained interactions
all_interactions = pd.concat(
    [train_80, validation_80, test_80],
    ignore_index=True
)

print("Training interactions:", len(train_80))
print("Validation interactions:", len(validation_80))
print("Test interactions:", len(test_80))

print("\nCombined interactions:", len(all_interactions))

print(
    "Duplicate user-activity pairs:",
    all_interactions.duplicated(
        subset=["user_id", "gmap_id"]
    ).sum()
)

print(
    "Missing timestamps:",
    all_interactions["time"].isna().sum()
)

print(
    "Unique users:",
    all_interactions["user_id"].nunique()
)

print(
    "Unique activities:",
    all_interactions["gmap_id"].nunique()
)

Training interactions: 7890582
Validation interactions: 887509
Test interactions: 885709

Combined interactions: 9663800
Duplicate user-activity pairs: 0
Missing timestamps: 0
Unique users: 627365
Unique activities: 69995


In [6]:
# List the notebooks in the project
notebooks_folder = project / "notebooks"

print("Available notebooks:\n")

for notebook_path in sorted(notebooks_folder.glob("*.ipynb")):
    print(notebook_path.name)

Available notebooks:

01_data_exploration.ipynb
02_ogechika_interaction_exploration.ipynb
03_ogechika_data_split.ipynb
bpr_baseline.ipynb
data_split_60_20_20.ipynb


In [7]:
import json

original_notebook_path = (
    project / "notebooks" / "03_ogechika_data_split.ipynb"
)

with open(original_notebook_path, "r", encoding="utf-8") as file:
    original_notebook = json.load(file)

# Display the code cells from the original splitting notebook
for cell_number, cell in enumerate(original_notebook["cells"], start=1):
    if cell["cell_type"] == "code":
        code = "".join(cell["source"])

        print(f"\n{'=' * 60}")
        print(f"CELL {cell_number}")
        print(f"{'=' * 60}")
        print(code[:3000])


CELL 1
import pandas as pd

df = pd.read_csv(
    "/Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/processed/activity_dataset.csv"
)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 3 rows:")
print(df.head(3).to_string())

print("\nData types:")
print(df.dtypes)


CELL 2
# Check how much activity information is available

columns = [
    "name",
    "gmap_id",
    "latitude",
    "longitude",
    "category",
    "activity_groups",
    "price",
    "hours",
    "description",
    "state"
]

audit = pd.DataFrame({
    "Available": df[columns].notna().sum(),
    "Missing": df[columns].isna().sum(),
    "Percent Available": (
        df[columns].notna().mean() * 100
    ).round(2)
})

print(audit.to_string())

print("\nPRICE VALUES:")
print(df["price"].value_counts(dropna=False).head(15))

print("\nOPERATING STATUS:")
print(df["state"].value_counts(dropna=False).head(15))

print("\nUNIQUE ACTIVITIES:")
print(df["gmap_i

In [8]:
import json
import pandas as pd

# Use the unique activities from the processed catalog
activity_ids = set(activity_df["gmap_id"].dropna())

matched_reviews = []
total_reviews = 0

with open(review_path, "r", encoding="utf-8") as file:
    for line in file:
        total_reviews += 1
        review = json.loads(line)

        if review.get("gmap_id") in activity_ids:
            matched_reviews.append({
                "user_id": review.get("user_id"),
                "gmap_id": review.get("gmap_id"),
                "rating": review.get("rating"),
                "time": review.get("time")
            })

reviews_df = pd.DataFrame(
    matched_reviews,
    columns=["user_id", "gmap_id", "rating", "time"]
)

# Reproduce the original cleaning rules
clean_reviews = reviews_df.dropna(
    subset=["user_id", "gmap_id", "rating", "time"]
).copy()

clean_reviews = clean_reviews.sort_values("time")

clean_reviews = clean_reviews.drop_duplicates(
    subset=["user_id", "gmap_id"],
    keep="last"
)

positive_interactions = clean_reviews[
    clean_reviews["rating"] >= 4
].copy()

print("Total raw reviews:", total_reviews)
print("Reviews matching activity catalog:", len(reviews_df))
print("Cleaned positive interactions:", len(positive_interactions))
print("Unique users:", positive_interactions["user_id"].nunique())
print("Unique activities:", positive_interactions["gmap_id"].nunique())


Total raw reviews: 18661975
Reviews matching activity catalog: 11790233
Cleaned positive interactions: 9665976
Unique users: 627365
Unique activities: 70178


In [9]:
import numpy as np

# Keep the same columns as the original experiment
data_60 = positive_interactions[
    ["user_id", "gmap_id", "rating", "time"]
].copy()

# Sort interactions chronologically for each user
data_60 = data_60.sort_values(
    ["user_id", "time", "gmap_id"]
).reset_index(drop=True)

# Count interactions per user
user_sizes = data_60.groupby("user_id")["gmap_id"].transform("size")

# Position of each interaction in the user's history
positions = data_60.groupby("user_id").cumcount()

# Users with at least 5 interactions are eligible for evaluation
eligible = user_sizes >= 5

# Allocate approximately 20% each to validation and test
n_val = np.where(
    eligible,
    np.maximum(1, np.floor(user_sizes * 0.20).astype(int)),
    0
)

n_test = np.where(
    eligible,
    np.maximum(1, np.floor(user_sizes * 0.20).astype(int)),
    0
)

# Remaining interactions go to training
n_train = user_sizes - n_val - n_test

# Assign chronological interactions
train_60 = data_60[
    positions < n_train
].copy()

validation_60 = data_60[
    (positions >= n_train) &
    (positions < n_train + n_val)
].copy()

test_60 = data_60[
    positions >= n_train + n_val
].copy()

print("INITIAL 60/20/20 SPLIT RESULTS")
print("Training:", len(train_60))
print("Validation:", len(validation_60))
print("Test:", len(test_60))

print("\nTOTAL:", len(train_60) + len(validation_60) + len(test_60))

print("\nACTUAL PERCENTAGES")
print(f"Training: {len(train_60) / len(data_60) * 100:.2f}%")
print(f"Validation: {len(validation_60) / len(data_60) * 100:.2f}%")
print(f"Test: {len(test_60) / len(data_60) * 100:.2f}%")

print("\nUNIQUE ACTIVITIES")
print("Training:", train_60["gmap_id"].nunique())
print("Validation:", validation_60["gmap_id"].nunique())
print("Test:", test_60["gmap_id"].nunique())

INITIAL 60/20/20 SPLIT RESULTS
Training: 6303944
Validation: 1681016
Test: 1681016

TOTAL: 9665976

ACTUAL PERCENTAGES
Training: 65.22%
Validation: 17.39%
Test: 17.39%

UNIQUE ACTIVITIES
Training: 69783
Validation: 67716
Test: 67433


In [10]:
# Identify activities present in the training set
train_activity_ids_60 = set(train_60["gmap_id"])

# Find validation and test interactions involving unseen activities
unseen_val_60 = ~validation_60["gmap_id"].isin(
    train_activity_ids_60
)

unseen_test_60 = ~test_60["gmap_id"].isin(
    train_activity_ids_60
)

print("COLD-START ACTIVITY CHECK")

print(
    "Validation interactions with unseen activities:",
    unseen_val_60.sum()
)

print(
    "Test interactions with unseen activities:",
    unseen_test_60.sum()
)

print("\nPERCENTAGE AFFECTED")

print(
    "Validation:",
    f"{unseen_val_60.mean() * 100:.4f}%"
)

print(
    "Test:",
    f"{unseen_test_60.mean() * 100:.4f}%"
)

print("\nUNSEEN UNIQUE ACTIVITIES")

print(
    "Validation:",
    validation_60.loc[
        unseen_val_60, "gmap_id"
    ].nunique()
)

print(
    "Test:",
    test_60.loc[
        unseen_test_60, "gmap_id"
    ].nunique()
)

COLD-START ACTIVITY CHECK
Validation interactions with unseen activities: 527
Test interactions with unseen activities: 4482

PERCENTAGE AFFECTED
Validation: 0.0314%
Test: 0.2666%

UNSEEN UNIQUE ACTIVITIES
Validation: 236
Test: 395


In [11]:
# Remove interactions involving activities absent from training
validation_60_final = validation_60[
    validation_60["gmap_id"].isin(train_activity_ids_60)
].copy()

test_60_final = test_60[
    test_60["gmap_id"].isin(train_activity_ids_60)
].copy()

# Check for overlapping user-activity pairs
def get_pairs(frame):
    return pd.MultiIndex.from_frame(
        frame[["user_id", "gmap_id"]]
    )

train_pairs_60 = get_pairs(train_60)
val_pairs_60 = get_pairs(validation_60_final)
test_pairs_60 = get_pairs(test_60_final)

assert len(train_pairs_60.intersection(val_pairs_60)) == 0
assert len(train_pairs_60.intersection(test_pairs_60)) == 0
assert len(val_pairs_60.intersection(test_pairs_60)) == 0

# Confirm all evaluation users appear in training
train_user_ids_60 = set(train_60["user_id"])

assert set(validation_60_final["user_id"]).issubset(
    train_user_ids_60
)

assert set(test_60_final["user_id"]).issubset(
    train_user_ids_60
)

# Confirm all evaluation activities appear in training
assert set(validation_60_final["gmap_id"]).issubset(
    train_activity_ids_60
)

assert set(test_60_final["gmap_id"]).issubset(
    train_activity_ids_60
)

# Confirm the expected number of interactions was removed
assert len(validation_60) - len(validation_60_final) == 527
assert len(test_60) - len(test_60_final) == 4482

print("All split validation checks passed!")

print("\nFINAL SPLIT SIZES")
print("Training:", len(train_60))
print("Validation:", len(validation_60_final))
print("Test:", len(test_60_final))

total_final_60 = (
    len(train_60)
    + len(validation_60_final)
    + len(test_60_final)
)

print("\nTotal retained interactions:", total_final_60)
print("Total removed interactions:", len(data_60) - total_final_60)

print("\nFINAL PERCENTAGES")
print(f"Training: {len(train_60) / total_final_60 * 100:.2f}%")
print(f"Validation: {len(validation_60_final) / total_final_60 * 100:.2f}%")
print(f"Test: {len(test_60_final) / total_final_60 * 100:.2f}%")

All split validation checks passed!

FINAL SPLIT SIZES
Training: 6303944
Validation: 1680489
Test: 1676534

Total retained interactions: 9660967
Total removed interactions: 5009

FINAL PERCENTAGES
Training: 65.25%
Validation: 17.39%
Test: 17.35%


In [12]:
# Create a separate directory for the 60/20/20 experiment
split_dir_60 = project / "data" / "splits_60_20_20"
split_dir_60.mkdir(parents=True, exist_ok=True)

# Save the final datasets
train_60.to_csv(
    split_dir_60 / "train.csv",
    index=False
)

validation_60_final.to_csv(
    split_dir_60 / "validation.csv",
    index=False
)

test_60_final.to_csv(
    split_dir_60 / "test.csv",
    index=False
)

print("60/20/20 datasets saved successfully!\n")

for filename in ["train.csv", "validation.csv", "test.csv"]:
    file_path = split_dir_60 / filename

    print(
        f"{filename}: "
        f"{file_path.stat().st_size / (1024**2):.2f} MB"
    )

print("\nSaved to:", split_dir_60)

60/20/20 datasets saved successfully!

train.csv: 456.49 MB
validation.csv: 121.69 MB
test.csv: 121.40 MB

Saved to: /Users/ogechikaesedebe/Downloads/Activity-Recommendation-System/data/splits_60_20_20
